In [16]:
import os
os.chdir('.')  # skip this if already run once and restarted from project root — see note below
print("Working from:", os.getcwd())

Working from: /home/zohaibtech92/messy-csv-cleanup


In [17]:
import pandas as pd
import numpy as np

In [18]:
wide_columns = [
    'UNITID', 'OPEID6', 'INSTNM', 'CITY', 'STABBR', 'ZIP',
    'PREDDEG', 'HIGHDEG', 'CONTROL', 'REGION', 'LOCALE', 'MAIN', 'NUMBRANCH',
    'ADM_RATE', 'SATVR25', 'SATVR75', 'SATMT25', 'SATMT75', 'ACTCM25', 'ACTCM75',
    'NPT4_PUB', 'NPT4_PRIV', 'COSTT4_A', 'TUITIONFEE_IN', 'TUITIONFEE_OUT',
    'UGDS', 'UG', 'PCTFLOAN', 'PCTPELL',
    'C150_4', 'C150_L4', 'RET_FT4', 'RET_PT4',
    'DEBT_MDN', 'GRAD_DEBT_MDN', 'DEFAULT_RATE', 'PPTUG_EF',
    'MD_EARN_WNE_P10', 'MN_EARN_WNE_P10', 'COUNT_WNE_P10',
]

df = pd.read_csv(
    'data/raw/MERGED2017_18_PP.csv',
    usecols=lambda c: c in wide_columns,
    na_values=['NULL', 'PrivacySuppressed']
)
print(df.shape)

(7112, 39)


In [19]:
df_filtered = df[(df['PREDDEG'] == 3) & (df['ADM_RATE'].notna())].copy()
print("After filter:", df_filtered.shape)

After filter: (1671, 39)


In [20]:
# Official ACT Composite -> SAT Total (combined) concordance points, 2018 table
act_points = [11, 14, 17, 20, 23, 26, 29, 32, 35, 36]
sat_points = [670, 800, 930, 1040, 1140, 1240, 1340, 1430, 1540, 1590]

# Step 1: build combined SAT scores (verbal + math) from the two halves
df_filtered['sat_combined_25'] = df_filtered['SATVR25'] + df_filtered['SATMT25']
df_filtered['sat_combined_75'] = df_filtered['SATVR75'] + df_filtered['SATMT75']

# Step 2: convert ACT composite scores to SAT-equivalent, as a Series matching df_filtered's index
act_converted_25 = pd.Series(
    np.interp(df_filtered['ACTCM25'], act_points, sat_points),
    index=df_filtered.index
)
act_converted_75 = pd.Series(
    np.interp(df_filtered['ACTCM75'], act_points, sat_points),
    index=df_filtered.index
)

# Step 3: track where each final score actually came from, BEFORE filling
df_filtered['test_score_source'] = np.where(
    df_filtered['sat_combined_25'].notna(), 'SAT',
    np.where(df_filtered['ACTCM25'].notna(), 'ACT_converted', 'missing')
)

# Step 4: fill missing SAT values with the converted ACT value, only where SAT was missing
df_filtered['sat_combined_25'] = df_filtered['sat_combined_25'].fillna(act_converted_25)
df_filtered['sat_combined_75'] = df_filtered['sat_combined_75'].fillna(act_converted_75)

# Step 5: drop rows where a test score is STILL missing after the combine attempt
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['sat_combined_25'].notna()].copy()
print(f"Dropped {before - len(df_filtered)} rows with no test score at all")
print(f"Remaining rows: {len(df_filtered)}")
print(df_filtered['test_score_source'].value_counts())

Dropped 384 rows with no test score at all
Remaining rows: 1287
test_score_source
SAT              1209
ACT_converted      78
Name: count, dtype: int64


In [21]:
# Verify: does CONTROL (school type) actually explain the split?
# CONTROL: 1 = Public, 2 = Private nonprofit, 3 = Private for-profit
print(df_filtered.groupby('CONTROL')[['NPT4_PUB', 'NPT4_PRIV']].apply(lambda x: x.notna().sum()))

         NPT4_PUB  NPT4_PRIV
CONTROL                     
1             494          0
2               0        777
3               0          5


In [22]:
df_filtered['net_price'] = df_filtered['NPT4_PUB'].fillna(df_filtered['NPT4_PRIV'])
print("net_price missing:", df_filtered['net_price'].isna().sum())

net_price missing: 11


In [23]:
# Which Group 4 columns still have any missingness at this point?
low_missing_cols = ['GRAD_DEBT_MDN', 'DEBT_MDN', 'PCTPELL', 'PCTFLOAN', 'PPTUG_EF',
                     'UGDS', 'LOCALE', 'C150_4', 'COSTT4_A',
                     'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'RET_FT4', 'net_price']

missing_check = df_filtered[low_missing_cols].isna().sum()
print(missing_check[missing_check > 0])

before = len(df_filtered)
df_filtered = df_filtered.dropna(subset=low_missing_cols).copy()
print(f"\nDropped {before - len(df_filtered)} rows ({(before - len(df_filtered))/before*100:.1f}%)")
print(f"Remaining rows: {len(df_filtered)}")

GRAD_DEBT_MDN     24
DEBT_MDN          17
PCTPELL            2
PCTFLOAN           2
C150_4            11
COSTT4_A          11
TUITIONFEE_IN      4
TUITIONFEE_OUT     4
RET_FT4            2
net_price         11
dtype: int64

Dropped 34 rows (2.6%)
Remaining rows: 1253


In [24]:
# Check for formatting inconsistencies in text columns
text_cols = ['INSTNM', 'CITY', 'STABBR']

for col in text_cols:
    print(f"--- {col} ---")
    # leading/trailing whitespace
    has_whitespace = df_filtered[col].str.strip().ne(df_filtered[col]).sum()
    print(f"Rows with leading/trailing whitespace: {has_whitespace}")
    # sample of unique values to eyeball casing/inconsistency
    print(df_filtered[col].unique()[:10])
    print()

--- INSTNM ---
Rows with leading/trailing whitespace: 0
<StringArray>
[           'Alabama A & M University', 'University of Alabama at Birmingham',
 'University of Alabama in Huntsville',            'Alabama State University',
           'The University of Alabama',     'Auburn University at Montgomery',
                   'Auburn University',         'Birmingham Southern College',
                 'Faulkner University',                  'Huntingdon College']
Length: 10, dtype: str

--- CITY ---
Rows with leading/trailing whitespace: 0
<StringArray>
[      'Normal',   'Birmingham',   'Huntsville',   'Montgomery',
   'Tuscaloosa',       'Auburn', 'Jacksonville',       'Marion',
   'Livingston',       'Mobile']
Length: 10, dtype: str

--- STABBR ---
Rows with leading/trailing whitespace: 0
<StringArray>
['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'DC', 'FL']
Length: 10, dtype: str



In [25]:
# How many ZIPs are 5-digit vs ZIP+4 format?
zip_lengths = df_filtered['ZIP'].str.len().value_counts()
print(zip_lengths)

ZIP
10    785
5     468
Name: count, dtype: int64


In [26]:
# Normalize ZIP to consistent 5-digit format
df_filtered['ZIP'] = df_filtered['ZIP'].str[:5]

# Verify: every value should now be exactly 5 characters
print(df_filtered['ZIP'].str.len().value_counts())

ZIP
5    1253
Name: count, dtype: int64
